# Neural Hydrology — A/B/C Publication Run (Colab)

This notebook runs the full publication-grade A/B/C ablation on the 183-basin **Component 0** network, per the locked protocol in `idea1.md`.

## What this does

Trains 15 models total (3 conditions × 5 seeds × 30 epochs) and writes results to Google Drive.

| Cond. | What | Seeds | Wall-clock per run |
|---|---|---|---|
| **A** | NH `cudalstm` + basin encoding (no graph) | 11/13/17/19/23 | ~10 min on A100 / ~30 min on T4 |
| **B** | DirectedGraph-LSTM, no edges, +5 topology features | 11/13/17/19/23 | ~15 min on A100 / ~30 min on T4 |
| **C** | DirectedGraph-LSTM, full edges + message passing | 11/13/17/19/23 | ~15 min on A100 / ~30 min on T4 |
| **Total** | | | **~3-4 hr A100 / ~7-8 hr T4** |

## How to use

1. Complete the one-time setup below (data + repo to Drive)
2. Set runtime to GPU (Runtime → Change runtime type → A100 / T4)
3. **Runtime → Run all** — the whole publication run executes
4. Results land in `/content/drive/MyDrive/neural_hydrology_runs/`
5. Pull locally + ask CRS to interpret (see final cell)

## One-time setup (do this before first run)

```bash
# On your local Mac:
cd /Users/om/Desktop/neural_hydrology

# 1. Zip the dataset and upload to Drive (one-time, ~10 GB)
tar -czf /tmp/camels_us.tar.gz datasets/camels_us
# Then upload /tmp/camels_us.tar.gz to MyDrive/neural_hydrology_data/

# 2. Zip the repo (excluding datasets and runs) and upload to Drive
tar -czf /tmp/nh_code.tar.gz \
    --exclude='datasets' --exclude='runs' --exclude='.git' \
    --exclude='__pycache__' --exclude='*.pt' \
    .
# Then upload /tmp/nh_code.tar.gz to MyDrive/neural_hydrology_data/
```

Alternative to zip: push the repo to a private GitHub and use the `git clone` cell instead.

## Cell 1 — Mount Drive + verify

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA = '/content/drive/MyDrive/neural_hydrology_data'
DRIVE_RUNS = '/content/drive/MyDrive/neural_hydrology_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)

assert os.path.isdir(DRIVE_DATA), (
    f'Expected {DRIVE_DATA} to exist with camels_us.tar.gz and nh_code.tar.gz inside. '
    'See the one-time-setup instructions in the markdown above.'
)
print('Drive mounted. Data dir:', DRIVE_DATA)
print('Files in data dir:')
!ls -lah {DRIVE_DATA}

## Cell 2 — Repo setup

Two options — uncomment the one you want.

In [ ]:
REPO_DIR = '/content/nh'

# === Option A: extract from Drive zip (no GitHub needed) ===
import shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
os.makedirs(REPO_DIR, exist_ok=True)
code_zip = os.path.join(DRIVE_DATA, 'nh_code.tar.gz')
assert os.path.isfile(code_zip), f'Missing {code_zip} — see one-time-setup'
!tar -xzf {code_zip} -C {REPO_DIR}

# === Option B: clone from your private GitHub (uncomment + set URL) ===
# GITHUB_REPO = 'https://github.com/<you>/neural_hydrology.git'
# !rm -rf {REPO_DIR}
# !git clone {GITHUB_REPO} {REPO_DIR}

%cd {REPO_DIR}
!ls

## Cell 3 — Install dependencies

Pinning `numpy<2` because torch 2.2 is incompatible with numpy 2.x (we hit this on the local environment).

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . "numpy<2" pynhd networkx pandas matplotlib scipy 2>&1 | tail -5

# Verify torch + numpy compatibility
import warnings
warnings.filterwarnings('error')
import numpy as np
import torch
x = torch.from_numpy(np.array([1.0, 2.0]))
print(f'numpy {np.__version__}  torch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')
warnings.resetwarnings()

## Cell 4 — Data sync (extract CAMELS-US once, symlink thereafter)

In [ ]:
%cd {REPO_DIR}
import os

# Where the data should live in the repo
REPO_DATA = os.path.join(REPO_DIR, 'datasets', 'camels_us')

# Persist data on Drive once-extracted (next session reuses it)
DRIVE_CAMELS = os.path.join(DRIVE_DATA, 'camels_us_extracted')
if not os.path.isdir(DRIVE_CAMELS):
    print('First-time extract of camels_us.tar.gz to Drive (slow, ~10 min)...')
    os.makedirs(DRIVE_CAMELS, exist_ok=True)
    camels_zip = os.path.join(DRIVE_DATA, 'camels_us.tar.gz')
    assert os.path.isfile(camels_zip), f'Missing {camels_zip} — see one-time-setup'
    !tar -xzf {camels_zip} -C {DRIVE_CAMELS} --strip-components=1
else:
    print(f'Reusing existing extracted data at {DRIVE_CAMELS}')

# Symlink into the repo
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
if os.path.islink(REPO_DATA) or os.path.isdir(REPO_DATA):
    !rm -rf {REPO_DATA}
os.symlink(DRIVE_CAMELS, REPO_DATA)

# Verify
import subprocess
n_basins_topo = subprocess.run(
    ['wc', '-l', f'{REPO_DATA}/camels_attributes_v2.0/camels_topo.txt'],
    capture_output=True, text=True).stdout
print(f'CAMELS topo file: {n_basins_topo.strip()}')

# Symlink runs/ to Drive (so trained models survive session end)
REPO_RUNS = os.path.join(REPO_DIR, 'runs')
if os.path.islink(REPO_RUNS) or os.path.isdir(REPO_RUNS):
    !rm -rf {REPO_RUNS}
os.symlink(DRIVE_RUNS, REPO_RUNS)
print(f'runs/ -> {DRIVE_RUNS}')

## Cell 5 — GPU check

In [ ]:
!nvidia-smi -L
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected. In Colab: Runtime -> Change runtime type -> select GPU')
print(f'\nGPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 6 — Generate per-seed YAML configs for Condition A

In [ ]:
%cd {REPO_DIR}
import os

SEEDS = [11, 13, 17, 19, 23]
CONFIG_DIR = os.path.join(REPO_DIR, 'experiments', 'configs', '_seed_configs')
os.makedirs(CONFIG_DIR, exist_ok=True)

BASE_CONFIG = open('experiments/configs/lstm_component0_baseline.yaml').read()

for seed in SEEDS:
    cfg = BASE_CONFIG
    cfg = cfg.replace('experiment_name: lstm_component0_baseline',
                       f'experiment_name: A_baseline_seed{seed}')
    cfg = cfg.replace('device: cpu', 'device: cuda:0')
    # Append seed if not present
    if 'seed:' not in cfg:
        cfg += f'\nseed: {seed}\n'
    out = os.path.join(CONFIG_DIR, f'A_seed{seed}.yaml')
    with open(out, 'w') as f:
        f.write(cfg)
    print(f'  wrote {out}')

print(f'\n{len(SEEDS)} configs ready in {CONFIG_DIR}')

## Cell 7 — Pre-flight smoke test

2-epoch run of Condition C (smallest variant) on the 23-basin pilot to catch any setup issues fast. ~3 min on GPU.

In [ ]:
%cd {REPO_DIR}
!python experiments/training/train_graph_component0.py \
    --variant warm \
    --seed 42 \
    --smoke-test \
    --no-warm-start \
    --basin-file experiments/basin_lists/study_network_basins.txt \
    --edge-file topology_analysis/phase1_network_discovery/outputs/study_network_edges.csv \
    --baseline-run runs/05_lstm_23basin_strong_baseline 2>&1 | tail -10
print('\n=== Smoke test complete. If you see DONE above, proceed to the publication runs.')

## Cell 8 — Condition A (5 seeds × NH cudalstm baseline)

~50 min on A100 / ~2.5 hr on T4. Each seed produces a `runs/A_baseline_seed{N}_*` dir.

In [ ]:
%cd {REPO_DIR}
import os
import glob
import time

for seed in SEEDS:
    # Skip if already complete
    existing = glob.glob(f'{REPO_DIR}/runs/A_baseline_seed{seed}_*/model_epoch030.pt')
    if existing:
        print(f'[skip] Condition A seed={seed} already complete: {existing[0]}')
        continue
    cfg = f'{CONFIG_DIR}/A_seed{seed}.yaml'
    print(f'\n=== Condition A — seed={seed} ===')
    t0 = time.time()
    !python neuralhydrology/nh_run.py train --config-file {cfg} 2>&1 | tail -3
    print(f'    seed {seed} took {(time.time() - t0)/60:.1f} min')

## Cell 9 — Condition B (5 seeds × topology-features-augmented LSTM, no message passing)

In [ ]:
%cd {REPO_DIR}
import glob
import time

# B and C need a Component-0 baseline run for their cfg / scaler / id_to_int.
# Use the most recent A run as the cfg source (its cudalstm config is identical
# in dataset/scaler terms to what B and C consume).
BASELINE_FOR_BC = sorted(glob.glob(f'{REPO_DIR}/runs/A_baseline_seed*/'))[0]
print(f'Using baseline-run for B/C cfg + scaler: {BASELINE_FOR_BC}')

for seed in SEEDS:
    existing = glob.glob(f'{REPO_DIR}/runs/graph_c0_topology_features_seed{seed}_*/test_metrics.csv')
    if existing:
        print(f'[skip] Condition B seed={seed} already complete')
        continue
    print(f'\n=== Condition B — seed={seed} ===')
    t0 = time.time()
    !python experiments/training/train_graph_component0.py \
        --variant topology_features \
        --seed {seed} \
        --no-warm-start \
        --epochs 30 \
        --baseline-run {BASELINE_FOR_BC} 2>&1 | tail -3
    print(f'    seed {seed} took {(time.time() - t0)/60:.1f} min')

## Cell 10 — Condition C (5 seeds × full graph-LSTM with edges + message passing)

In [ ]:
%cd {REPO_DIR}
import glob
import time

for seed in SEEDS:
    existing = glob.glob(f'{REPO_DIR}/runs/graph_c0_warm_seed{seed}_*/test_metrics.csv')
    if existing:
        print(f'[skip] Condition C seed={seed} already complete')
        continue
    print(f'\n=== Condition C — seed={seed} ===')
    t0 = time.time()
    !python experiments/training/train_graph_component0.py \
        --variant warm \
        --seed {seed} \
        --no-warm-start \
        --epochs 30 \
        --baseline-run {BASELINE_FOR_BC} 2>&1 | tail -3
    print(f'    seed {seed} took {(time.time() - t0)/60:.1f} min')

## Cell 11 — Aggregate analysis

In [ ]:
%cd {REPO_DIR}
import glob
import json
from pathlib import Path
import numpy as np
import pandas as pd

def load_test_metrics(pattern):
    """Load all matching test_metrics.csv files; return list of {seed: dict}."""
    out = {}
    for p in sorted(glob.glob(pattern)):
        # extract seed from path
        import re
        m = re.search(r'seed(\d+)', p)
        seed = int(m.group(1)) if m else None
        df = pd.read_csv(p, dtype={'basin': str})
        out[seed] = df.set_index('basin')['NSE'].to_dict()
    return out

results_A = load_test_metrics(f'{REPO_DIR}/runs/A_baseline_seed*/test/model_epoch030/test_metrics.csv')
results_B = load_test_metrics(f'{REPO_DIR}/runs/graph_c0_topology_features_seed*/test_metrics.csv')
results_C = load_test_metrics(f'{REPO_DIR}/runs/graph_c0_warm_seed*/test_metrics.csv')

summary = {}
for label, results in [('A_baseline', results_A), ('B_topology_features', results_B), ('C_graph_messages', results_C)]:
    if not results:
        continue
    medians = [np.median(list(d.values())) for d in results.values()]
    summary[label] = {
        'n_seeds': len(medians),
        'median_NSE_per_seed': sorted(medians),
        'cross_seed_median': float(np.median(medians)),
        'cross_seed_std': float(np.std(medians)),
        'cross_seed_min': float(np.min(medians)),
        'cross_seed_max': float(np.max(medians)),
    }

print(json.dumps(summary, indent=2))

# Save summary to Drive for the local CRS step
OUT_DIR = Path(REPO_DIR) / 'experiments' / 'analysis_outputs' / 'abc_publication'
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Also write per-seed per-basin NSEs as one wide CSV for downstream analysis
rows = []
for label, results in [('A', results_A), ('B', results_B), ('C', results_C)]:
    for seed, basin_nse in results.items():
        for basin, nse in basin_nse.items():
            rows.append({'condition': label, 'seed': seed, 'basin': basin, 'NSE': nse})
if rows:
    df_all = pd.DataFrame(rows)
    df_all.to_csv(OUT_DIR / 'per_basin_per_seed.csv', index=False)
    print(f'\nWrote: {OUT_DIR / "per_basin_per_seed.csv"} ({len(rows)} rows)')

# Show headline
if {'A_baseline', 'C_graph_messages'}.issubset(summary):
    a = summary['A_baseline']['cross_seed_median']
    c = summary['C_graph_messages']['cross_seed_median']
    print(f'\n*** HEADLINE *** C - A median NSE delta = {c - a:+.3f}')
    if 'B_topology_features' in summary:
        b = summary['B_topology_features']['cross_seed_median']
        print(f'                B - A = {b - a:+.3f}')
        print(f'                C - B = {c - b:+.3f}  <-- the message-passing-helps margin')

## Cell 12 — Send results back for CRS interpretation

**Option A (Google Drive desktop sync):** if you have Drive desktop installed, the files are already on your Mac at `~/Google Drive/My Drive/neural_hydrology_runs/...`. Drag the `experiments/analysis_outputs/abc_publication/` folder into your local repo, then say `crs` in chat.

**Option B (git push back):** uncomment the cell below — pushes the result CSVs + JSON back to your repo so a `git pull` locally is enough.

After you have the result files locally, send a one-line message like *"crs interpret abc results"* and I'll read the summary, position it against the framing, and propose the next step.

In [ ]:
# # === Option B: push only the lightweight result files back to the repo ===
# # Requires that you cloned via Option B (GitHub) in Cell 2.
# %cd {REPO_DIR}
# !git config user.email "colab@example.com"
# !git config user.name "colab"
# !git checkout -b colab-abc-results-$(date +%Y%m%d-%H%M%S)
# !git add experiments/analysis_outputs/abc_publication/
# !git commit -m "A/B/C publication run results from Colab"
# !git push -u origin HEAD

print('Result files for the next CRS step:')
print(f'  {REPO_DIR}/experiments/analysis_outputs/abc_publication/summary.json')
print(f'  {REPO_DIR}/experiments/analysis_outputs/abc_publication/per_basin_per_seed.csv')
print()
print('These also live on Drive at:')
print(f'  /content/drive/MyDrive/neural_hydrology_runs/  (full run dirs)')
print()
print('Next: pull these to your local Mac repo and ask CRS to interpret.')